#Introduction to GPU Acceleration
### 🔍 Why Use GPUs?

GPUs are optimized for large-scale parallel computation, making them ideal for matrix-heavy tasks in deep learning. In this lab, you'll compare training times and performance on CPU vs GPU and learn how to write GPU-efficient code.


##Check device availability

## TensorFlow

In [1]:
import tensorflow as tf
print("Is GPU available?", tf.config.list_physical_devices('GPU'))

Is GPU available? [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


##PyTorch

In [2]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


## 🚀 Moving Models and Data to GPU

To fully utilize the GPU, both the model and input data must be moved to the GPU device. This ensures the computation is performed on the GPU instead of the CPU.

Let's see how to do this in both TensorFlow and PyTorch.


##TensorFlow - Using GPU Automatically

In [3]:
# TensorFlow uses GPU by default when available
import tensorflow as tf

with tf.device('/GPU:0'):  # or '/CPU:0' for CPU
    a = tf.random.normal([1000, 1000])
    b = tf.random.normal([1000, 1000])
    c = tf.matmul(a, b)
    print("Operation completed on:", c.device)


Operation completed on: /job:localhost/replica:0/task:0/device:GPU:0


##PyTorch - Manual GPU Transfer

In [4]:
import torch

# Use 'cuda' if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Example tensor operation on GPU
a = torch.randn(1000, 1000).to(device)
b = torch.randn(1000, 1000).to(device)
c = torch.matmul(a, b)

print("Tensor 'c' is on device:", c.device)


Tensor 'c' is on device: cuda:0


##Measuring Training Time on CPU vs GPU

## ⏱️ Performance Benchmark: CPU vs GPU

We'll train a simple model on the MNIST dataset using both CPU and GPU. This will help us visualize the speedup provided by GPU acceleration.

Steps:
- Train on CPU and measure the time
- Train on GPU and measure the time
- Compare the difference


In [5]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Data
transform = transforms.ToTensor()
train_data = datasets.MNIST(root='data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# Model
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = F.relu(self.fc1(x))
        return F.log_softmax(self.fc2(x), dim=1)




100%|██████████| 9.91M/9.91M [00:02<00:00, 4.94MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 129kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.3MB/s]


##Train on CPU

In [6]:
# Train on CPU
def train_on_cpu():
    device = torch.device("cpu")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ CPU training time: {end_time - start_time:.2f} sec")

train_on_cpu()

✅ CPU training time: 10.18 sec


##Train on GPU
Change your runtine to T4 GPU and run the following code block

In [7]:
# Train on GPU
def train_on_gpu():
    if not torch.cuda.is_available():
        print("🚫 CUDA not available on this system.")
        return

    device = torch.device("cuda")
    model = SimpleModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.NLLLoss()

    start_time = time.time()
    for epoch in range(1):  # Short training
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            output = model(images)
            loss = criterion(output, labels)
            loss.backward()
            optimizer.step()
    end_time = time.time()
    print(f"✅ GPU training time: {end_time - start_time:.2f} sec")

train_on_gpu()


✅ GPU training time: 8.87 sec


I suppose we got lesser time for the gpu, it makes more difference on larger models that have more number of layers and filters, gpu speeds up the matrix multiplication due to the presence of numerous small cores.

**So here's an activity for you**
##Use tensorflow to train a model on the MNIST digits dataset on both gpu and cpu and examine which one works faster.

Use your custom number of layers and filters to experiment with the hyperparameters of the model.

In [9]:
import tensorflow as tf
import time

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense

In [10]:
print("TensorFlow version:", tf.__version__)

gpu = tf.config.list_physical_devices('GPU')

print("Available GPU:", gpu)

TensorFlow version: 2.20.0
Available GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [11]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

print(x_train.shape)
print(x_test.shape)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
(60000, 28, 28)
(10000, 28, 28)


In [12]:
x_train = x_train.reshape(-1,28,28,1)
x_test = x_test.reshape(-1,28,28,1)

x_train = x_train.astype("float32") / 255
x_test = x_test.astype("float32") / 255

In [13]:
def create_model():

    model = Sequential()

    # Layer 1
    model.add(
        Conv2D(
            32,
            (3,3),
            activation='relu',
            input_shape=(28,28,1)
        )
    )

    model.add(MaxPooling2D())

    # Layer 2
    model.add(
        Conv2D(
            64,
            (3,3),
            activation='relu'
        )
    )

    model.add(MaxPooling2D())

    # Classification layers
    model.add(Flatten())

    model.add(
        Dense(
            128,
            activation='relu'
        )
    )

    model.add(
        Dense(
            10,
            activation='softmax'
        )
    )


    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

In [14]:
def train_cpu():

    with tf.device('/CPU:0'):

        model = create_model()

        start = time.time()

        history = model.fit(
            x_train,
            y_train,
            epochs=5,
            batch_size=128,
            verbose=1
        )

        end = time.time()

    cpu_time = end-start

    print(
        "CPU Training Time:",
        round(cpu_time,2),
        "seconds"
    )

    return model, cpu_time


cpu_model, cpu_time = train_cpu()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 68s 139ms/step - accuracy: 0.9426 - loss: 0.1966
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 81s 139ms/step - accuracy: 0.9838 - loss: 0.0522
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 65s 138ms/step - accuracy: 0.9888 - loss: 0.0381
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 82s 139ms/step - accuracy: 0.9917 - loss: 0.0272
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 82s 139ms/step - accuracy: 0.9927 - loss: 0.0225
CPU Training Time: 395.0 seconds


In [16]:
def train_gpu():

    if not tf.config.list_physical_devices('GPU'):
        print("GPU not available")
        return None, None
    with tf.device('/GPU:0'):
        model = create_model()
        start = time.time()
        history = model.fit(
            x_train,
            y_train,
            epochs=5,
            batch_size=128,
            verbose=1)
        end = time.time()
    gpu_time = end-start

    print(
        "GPU Training Time:",
        round(gpu_time,2),
        "seconds")
    return model, gpu_time
gpu_model, gpu_time = train_gpu()

Epoch 1/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.9412 - loss: 0.2046
Epoch 2/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9816 - loss: 0.0583
Epoch 3/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9876 - loss: 0.0406
Epoch 4/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.9904 - loss: 0.0308
Epoch 5/5
469/469 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9920 - loss: 0.0253
GPU Training Time: 14.93 seconds


In [17]:
print("CPU Time:", round(cpu_time,2), "seconds")
print("GPU Time:", round(gpu_time,2), "seconds")

print(
    "GPU Speedup:",
    round(cpu_time/gpu_time,2),
    "times faster")

CPU Time: 395.0 seconds
GPU Time: 14.93 seconds
GPU Speedup: 26.46 times faster


In [22]:
with tf.device('/CPU:0'):
    cpu_loss, cpu_accuracy = cpu_model.evaluate(
        x_test,
        y_test,
        verbose=0
    )

print("CPU Accuracy:", cpu_accuracy)
print("CPU Loss:", cpu_loss)

CPU Accuracy: 0.9911999702453613
CPU Loss: 0.027997976168990135


In [20]:
gpu_loss, gpu_accuracy = gpu_model.evaluate(
    x_test,
    y_test
)

print("GPU Accuracy:", gpu_accuracy)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9904 - loss: 0.0305
GPU Accuracy: 0.9904000163078308


In [23]:
Conv2D(16,(3,3))
Dense(64)

<Dense name=dense_6, built=False>

In [24]:
Conv2D(32,(3,3))
Conv2D(64,(3,3))
Dense(128)

<Dense name=dense_7, built=False>